# 04 — The collapse of Aculeo Lake

[Open in Colab](https://colab.research.google.com/github/NTU-CompHydroMet-Lab/AguaTrack-ARCO-SA-Tutorial/blob/main/notebooks/04_aculeo_lake_collapse.ipynb)

**What we're investigating.** Aculeo Lake in central Chile, once a popular
weekend resort, disappeared completely around 2018 — one of the most
visible recent examples of catchment-scale drying. Using AguaTrack's full
1990–2019 record we ask whether the *moisture source* feeding this
catchment shifted over the three decades preceding the lake's
disappearance. Specifically: did the local **continental-recycling
ratio** (the fraction of rainfall sourced from regional
evapotranspiration rather than imported from the ocean) trend
downwards?

**What you'll get.** Two figures and an inline trend statistics block:

- `fig1_recycling_timeseries.png`: yearly recycling-ratio bars + an
  ordinary least-squares trend line (with p-value) + an orange dashed
  line at the disappearance year.
- `fig2_period_comparison.png`: source-map composites for the early
  period (1990–2004) and the late period (2005–2019), shared colour
  bar.
- Trend statistics: OLS slope, p-value, period means, and relative
  change.

**Dataset.** AguaTrack v1, **yearly aggregate** zarr stores
(`{year}_yearly_sum.zarr`). Single tag cell at the lake's location.

**How to cite.** TODO: paper in submission.

## Step 1 — Configuration

Everything you might want to edit lives in this single cell:

- **HuggingFace dataset** — `AguaTrack/AguaTrack-ARCO-SA`. Yearly
  aggregate sub-directory is not yet uploaded.
- **Target location** — Aculeo Lake by default. Set `TARGET_LAT` /
  `TARGET_LON` to any other catchment with documented hydrological
  collapse to re-run the trend analysis there.
- **Period split** — 15 years early vs 15 years late. Long enough to
  smooth year-to-year noise, short enough to detect a regime shift.
- **Disappearance year** — used only as a vertical line annotation
  on the time-series figure.

In [ ]:
HF_REVISION = "main"

TARGET_NAME = "Aculeo Lake"
TARGET_LAT = -33.83
TARGET_LON = -71.0
ALL_YEARS = list(range(1990, 2020))
EARLY_PERIOD = list(range(1990, 2005))
LATE_PERIOD = list(range(2005, 2020))
DISAPPEARANCE_YEAR = 2018

# Full zarr URLs, one per year. The yearly-aggregate sub-directory is
# not yet on HuggingFace — these URLs will start resolving once it is
# uploaded under the same naming scheme.
AGUATRACK_YEARLY_URLS = [
    f"hf://datasets/AguaTrack/AguaTrack-ARCO-SA/AguaTrack_ARCO_SA_yearly/{yr}_yearly_sum.zarr"
    for yr in ALL_YEARS
]

## Step 2 — Install dependencies (Colab only)

In [ ]:
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from IPython import get_ipython
    get_ipython().run_line_magic(
        "pip",
        'install -q cartopy cmcrameri "xarray>=2026" "zarr>=3" '
        "fsspec huggingface_hub dask scipy",
    )

## Step 3 — Imports and plotting style

In [ ]:
from pathlib import Path

import cartopy.crs as ccrs
import cmcrameri.cm as cmc
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import xarray as xr
from scipy import stats as sp_stats

plt.rcParams.update({
    "font.size": 16,
    "axes.titlesize": 16,
    "axes.labelsize": 16,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 12,
})

## Step 4 — Find the tag cell nearest to Aculeo Lake

In [ ]:
ds_ref = xr.open_zarr(AGUATRACK_YEARLY_URLS[0],
                      storage_options={"revision": HF_REVISION})
# Euclidean argmin is fine at 0.25 deg resolution. No need for
# great-circle distance. argmin() returns a 0-D DataArray; cast to int.
dist_sq = (ds_ref.tag_lat - TARGET_LAT) ** 2 + (ds_ref.tag_lon - TARGET_LON) ** 2
tag_idx = int(dist_sq.argmin())
print(
    f"target=({TARGET_LAT}, {TARGET_LON})  "
    f"nearest=({float(ds_ref.tag_lat.isel(tagging_mask=tag_idx)):.2f}, "
    f"{float(ds_ref.tag_lon.isel(tagging_mask=tag_idx)):.2f})  idx={tag_idx}"
)

## Step 5 — Build a 30-year lazy graph and materialise once

Pangeo idiom: stack the 30 yearly stores into one virtual dataset
along a new `year` dim, build the lazy per-year source map and
recycling ratio, and trigger a single `.compute()` at the end with a
progress bar. dask schedules the 30 reads concurrently — much faster
than a sequential Python loop.

In [ ]:
from dask.diagnostics import ProgressBar

# Stack 30 lazy datasets along a new "year" dim. The single tag is
# selected before any data is read.
ds_all = xr.concat(
    [
        xr.open_zarr(url, storage_options={"revision": HF_REVISION})
            .isel(tagging_mask=tag_idx)
        for url in AGUATRACK_YEARLY_URLS
    ],
    dim=xr.Variable("year", ALL_YEARS),
)
lsm = ds_ref.lsm

# Lazy: per-year 2-D source map (mean over the singleton time axis).
year_src_lazy = ds_all.e_track.mean(dim="time")                       # (year, lat, lon)
# Lazy: continental-recycling ratio = land sum / total sum.
total_annual = year_src_lazy.sum(dim=("latitude", "longitude"))
land_annual = year_src_lazy.where(lsm).sum(dim=("latitude", "longitude"))
year_ratios_lazy = land_annual / total_annual.where(total_annual > 0)

# Materialise both in one go.
with ProgressBar():
    year_src = year_src_lazy.load()
    year_ratios = year_ratios_lazy.load()

ds_ref.close()
print(f"loaded {year_src.sizes['year']} years "
      f"({year_src.nbytes / 1e6:.1f} MB in RAM)")

## Step 6 — Annual time series with OLS trend

Bars are annual recycling ratios in %. The dark red line is an
ordinary-least-squares fit of recycling ratio vs year. The legend
reports slope (in %/yr) and the p-value of the slope, computed via
`scipy.stats.linregress`. The orange dashed vertical line marks the
year Aculeo disappeared, for visual context.

In [ ]:
# year_ratios is a DataArray indexed by `year`. Convert to %.
ratios_pct = year_ratios * 100
years_arr = ratios_pct.year.values.astype(float)
ratios = ratios_pct.values

# OLS via xarray polyfit. Returns coefficients on a `degree` axis
# (deg=1 -> [intercept, slope] for degree=[0, 1]).
fit = ratios_pct.polyfit(dim="year", deg=1)
slope = float(fit.polyfit_coefficients.sel(degree=1))
intercept = float(fit.polyfit_coefficients.sel(degree=0))
trend = slope * years_arr + intercept
# scipy gives us a p-value for the slope from the linear regression.
_, _, _, p_val, _ = sp_stats.linregress(years_arr, ratios)

fig, ax = plt.subplots(figsize=(16, 6.7))
ax.bar(ratios_pct.year.values, ratios, color="steelblue", alpha=0.75,
       label="Annual recycling ratio")
ax.plot(ratios_pct.year.values, trend, color="darkred", lw=2,
        label=f"OLS trend: {slope:+.3f} %/yr  (p={p_val:.4f})")
ax.axvline(DISAPPEARANCE_YEAR, color="orange", lw=1.8, linestyle="--",
           label=f"{TARGET_NAME} disappearance (~{DISAPPEARANCE_YEAR})")
ax.set_xlabel("Year")
ax.set_ylabel("Local Recycling Ratio (%)")
ax.set_title(f"Local Evaporative Recycling Ratio — {TARGET_NAME} (1990–2019)")
ax.legend(fontsize=11)
ax.set_xlim(1989, 2020)

OUT = Path("outputs/aculeo"); OUT.mkdir(parents=True, exist_ok=True)
fig.savefig(OUT / "fig1_recycling_timeseries.png", bbox_inches="tight", dpi=150)
plt.show()

## Step 7 — Early vs late period source map comparison

Both panels share a colour bar so source-strength is directly
comparable. The shapefile basin overlay is intentionally left out —
the focus is on the source field, not on basin boundaries.

In [ ]:
# Period composites via xarray slicing on the `year` coordinate.
early_map = year_src.sel(year=EARLY_PERIOD).mean(dim="year")
late_map = year_src.sel(year=LATE_PERIOD).mean(dim="year")
# Shared colour bar across periods. xarray reductions handle NaN.
vmax = float(xr.concat([early_map, late_map], dim="period").max())
levels = np.linspace(0, vmax, 11)


def style_ax(ax, title):
    """Decorate a single map panel."""
    ax.coastlines(resolution="50m", color="black", linewidth=0.8)
    # Zoom into southern South America — Aculeo is at ~33 deg S, the
    # source pixels are mostly in this window.
    ax.set_extent([-85, -40, -55, -20], crs=ccrs.PlateCarree())
    gl = ax.gridlines(draw_labels=True, linewidth=0.0)
    gl.top_labels = gl.right_labels = False
    gl.xlocator = mticker.FixedLocator(np.arange(-180, 181, 10))
    gl.ylocator = mticker.FixedLocator(np.arange(-90, 91, 10))
    ax.set_title(title, fontsize=12)


fig, axes = plt.subplots(
    1, 2, figsize=(16, 8.0), subplot_kw={"projection": ccrs.PlateCarree()},
)
cf = None
pairs = [
    (early_map, f"Early Period ({EARLY_PERIOD[0]}–{EARLY_PERIOD[-1]})"),
    (late_map, f"Late Period ({LATE_PERIOD[0]}–{LATE_PERIOD[-1]})"),
]
for ax, (dmap, title) in zip(axes, pairs):
    cf = dmap.plot.contourf(
        ax=ax, levels=levels, cmap=cmc.batlowW,
        transform=ccrs.PlateCarree(), add_colorbar=False,
    )
    ax.scatter(
        TARGET_LON, TARGET_LAT, color="red", s=120, marker="*",
        transform=ccrs.PlateCarree(), zorder=5, label=TARGET_NAME,
    )
    ax.legend(loc="lower left", fontsize=10)
    style_ax(ax, title)

# Shared horizontal colour bar.
fig.subplots_adjust(bottom=0.12, wspace=0.05)
cbar_ax = fig.add_axes([0.15, 0.05, 0.7, 0.02])
fig.colorbar(cf, cax=cbar_ax, orientation="horizontal",
             label="Mean Moisture Contribution (mm/yr)")
fig.suptitle(f"Moisture Source Shift — {TARGET_NAME}", fontsize=14)
fig.savefig(OUT / "fig2_period_comparison.png", bbox_inches="tight", dpi=150)
plt.show()

## Step 8 — Trend statistics summary

In [ ]:
early_mean = float(ratios_pct.sel(year=EARLY_PERIOD).mean())
late_mean = float(ratios_pct.sel(year=LATE_PERIOD).mean())
# Relative change between period means, expressed as a percentage of the
# early-period mean.
pct_change = (late_mean - early_mean) / early_mean * 100

print(f"Trend statistics for {TARGET_NAME} recycling ratio (1990–2019):\n")
print(f"  OLS slope      : {slope:+.4f} %/yr  ({slope*10:+.3f} %/decade)")
print(f"  p-value        : {p_val:.4f}")
print(f"  Early mean     : {early_mean:.2f} %  ({EARLY_PERIOD[0]}–{EARLY_PERIOD[-1]})")
print(f"  Late mean      : {late_mean:.2f} %  ({LATE_PERIOD[0]}–{LATE_PERIOD[-1]})")
print(f"  Relative change: {pct_change:+.1f} %")

## Try another catchment

To compare other regions with documented hydrological collapse, copy
this notebook and change `TARGET_LAT` / `TARGET_LON` in Step 3.